# Eval Feedback Review
Compare human reviewer scores/feedback against automated eval scorers, identify discrepancies, and suggest prompt improvements.

## Setup

In [1]:
from legal_rag.eval_review import EvalReviewClient
from config import *

import os
import pandas as pd

In [ ]:
EVAL_RESULTS_PATH = os.path.join(EVAL_RESULTS_ROOT_PATH, "Gemini-3.0 Flash Legal V02.json")
EVAL_RESULTS_PATH

'docs/KKP/LNC/eval_results/Gemini-3.0 Flash Legal V02.json'

## 1. Initialize Review Client

In [3]:
reviewer = EvalReviewClient(
    eval_model=EVAL_AI_MODEL,
    review_model=REVIEW_AI_MODEL,
    eval_prompt_version=EVAL_AI_PROMPT_VERSION,
    prompt_version=REVIEW_AI_PROMPT_VERSION,
)
reviewer

## 2. Load & Transform Eval Results JSON

In [ ]:
data = EvalReviewClient.load_eval_json(EVAL_RESULTS_PATH)

print(f"Loaded {len(data)} entries\n")
for i, d in enumerate(data):
    print(f"Entry {i+1}: {d['input'][:80]}...")
    print(f"  Auto:  {d['auto_scores']}")
    print(f"  Human: {d['human_scores']}")
    print("-------------------------------")

Loaded 5 entries

Entry 1: เจ้าของบัญชีเงินฝากถึงแก่กรรม บุคคลที่อ้างว่าเป็นทายาทขอดำเนินการจัดการบัญชี เช่...
  Auto:  {'reference': 0, 'judgement': 0, 'suggestion': 0}
  Human: {'reference': 0.5, 'judgement': 0.5, 'suggestion': 0}

Entry 2: นิติบุคคลเปิดบัญชีเงินฝาก ใช้หนังสือแสดงความประสงค์เปิดบัญชีแล้วลงนามโดยกรรมการผ...
  Auto:  {'reference': 0.5, 'judgement': 0, 'suggestion': 0}
  Human: {'reference': 0.5, 'judgement': 0.5, 'suggestion': 0.5}

Entry 3: เจ้าของบัญชีเงินฝากที่เสียชีวิตเป็นคนต่างชาติ ผู้ที่อ้างว่าเป็นทายาทมาขอถอนเงิน ...
  Auto:  {'reference': 0.5, 'judgement': 0.5, 'suggestion': 1}
  Human: {'reference': 0.5, 'judgement': 1, 'suggestion': 0.5}

Entry 4: ผู้เยาว์สามารถเปิดบัญชีเงินฝากกับธนาคารได้หรือไม่...
  Auto:  {'reference': 0.5, 'judgement': 1, 'suggestion': 1}
  Human: {'reference': 0, 'judgement': 0, 'suggestion': 1}

Entry 5: กรณีลูกค้าขอเบิกถอนเงินฝากประจำ ธนาคารไม่ยินยอมให้ถอนก่อนครบกำหนดระยะเวลาฝากของล...
  Auto:  {'reference': 1, 'judgement': 1, 'suggest

In [5]:
# Quick score comparison table
rows = []
for i, d in enumerate(data):
    for key in ["reference", "judgement", "suggestion"]:
        rows.append({
            "entry": i + 1,
            "scorer": key,
            "auto_score": d["auto_scores"][key],
            "human_score": d["human_scores"][key],
            "diff": d["auto_scores"][key] - d["human_scores"][key],
        })

score_df = pd.DataFrame(rows)
score_df

,entry,scorer,auto_score,human_score,diff
0,1,reference,0.0,0.5,-0.5
1,1,judgement,0.0,0.5,-0.5
2,1,suggestion,0.0,0.0,0.0
3,2,reference,0.5,0.5,0.0
4,2,judgement,0.0,0.5,-0.5
5,2,suggestion,0.0,0.5,-0.5
6,3,reference,0.5,0.5,0.0
7,3,judgement,0.5,1.0,-0.5
8,3,suggestion,1.0,0.5,0.5
9,4,reference,0.5,0.0,0.5


## 3. Re-run Scorers (with rationale capture)

In [5]:
reviews = reviewer.rerun_scorers(data)

/Users/rit/Documents/GitHub/Study RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [1/5] reference: 0.5
  [1/5] judgement: 0.0
  [1/5] suggestion: 1.0
  [2/5] reference: 0.5
  [2/5] judgement: 0.0
  [2/5] suggestion: 0.0
  [3/5] reference: 0.5
  [3/5] judgement: 0.5
  [3/5] suggestion: 1.0
  [4/5] reference: 0.5
  [4/5] judgement: 1.0
  [4/5] suggestion: 0.5
  [5/5] reference: 1.0
  [5/5] judgement: 1.0
  [5/5] suggestion: 1.0


In [6]:
# Inspect re-run results for first entry
for sr in reviews[0].scorer_results:
    print(f"--- {sr.scorer_name} ---")
    print(f"Score: {sr.score}")
    print(f"Choice: {sr.choice}")
    print(f"Rationale: {sr.rationale[:300]}...")
    print()

--- reference ---
Score: 0.5
Choice: อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น
Rationale: การประเมินเปรียบเทียบระหว่าง OUTPUT และ EXPECTED ในส่วนที่ 1 (มาตรากฎหมายที่เกี่ยวข้อง) มีรายละเอียดดังนี้:

1. **วิเคราะห์เฉลย (EXPECTED):** ระบุมาตราหลักคือ ป.พ.พ. มาตรา 1713 (เหตุในการตั้งผู้จัดการมรดก), 1718 (คุณสมบัติผู้จัดการมรดก), และ 1719 (สิทธิหน้าที่ผู้จัดการมรดก) ซึ่งเน้นไปที่กระบวนการตั้...

--- judgement ---
Score: 0.0
Choice: ไม่ถูกต้องตามเฉลย
Rationale: ผลการประเมินคือ 'ไม่ถูกต้องตามเฉลย' โดยมีเหตุผลประกอบดังนี้:

1. **ความขัดแย้งในผลทางกฎหมาย (Conclusion Mismatch):**
   - **เฉลย (EXPECTED):** วินิจฉัยว่าธนาคาร **ต้อง** อาศัยการดำเนินการของผู้จัดการมรดกตามคำสั่งศาลเท่านั้น เนื่องจากธนาคารไม่สามารถทราบได้ว่าทายาทที่มาติดต่อเป็นทายาททั้งหมดหรือไม่ เพ...

--- suggestion ---
Score: 1.0
Choice: ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง
Rationale: คำตอบ (OUTPUT) มีความถูกต้อง ชัดเจน และครอบคลุมมากกว่าเฉลย (EXPECTED) โดยมีเหตุผลดังนี้:

1.  **ความถูกต้อง

In [7]:
# Compare re-run scores vs original auto scores vs human scores
rerun_rows = []
for review in reviews:
    for sr in review.scorer_results:
        rerun_rows.append({
            "entry": review.index + 1,
            "scorer": sr.scorer_name,
            "rerun_score": sr.score,
            "original_auto": review.auto_scores.get(sr.scorer_name, None),
            "human_score": review.human_scores.get(sr.scorer_name, None),
        })

rerun_df = pd.DataFrame(rerun_rows)
rerun_df

,entry,scorer,rerun_score,original_auto,human_score
0,1,reference,0.5,0.0,0.5
1,1,judgement,0.0,0.0,0.5
2,1,suggestion,1.0,0.0,0.0
3,2,reference,0.5,0.5,0.5
4,2,judgement,0.0,0.0,0.5
5,2,suggestion,0.0,0.0,0.5
6,3,reference,0.5,0.5,0.5
7,3,judgement,0.5,0.5,1.0
8,3,suggestion,1.0,1.0,0.5
9,4,reference,0.5,0.5,0.0


## 4. Summarize Discrepancies (Human Feedback vs Auto Rationale)

In [8]:
reviews = reviewer.summarize_discrepancies(reviews)

  [1/5] Discrepancy summary done
  [2/5] Discrepancy summary done
  [3/5] Discrepancy summary done
  [4/5] Discrepancy summary done
  [5/5] Discrepancy summary done


In [9]:
# View discrepancy summary for each entry
for review in reviews:
    print(f"=== Entry {review.index + 1} ===")
    print(f"Q: {review.input[:100]}...")
    print(f"\nDiscrepancy Summary:\n{review.discrepancy_summary}")
    print("\n" + "="*80 + "\n")

=== Entry 1 ===
Q: เจ้าของบัญชีเงินฝากถึงแก่กรรม บุคคลที่อ้างว่าเป็นทายาทขอดำเนินการจัดการบัญชี เช่น ปิดบัญชีเงินฝากดัง...

Discrepancy Summary:
นี่คือบทสรุปการเปรียบเทียบการประเมินระหว่างผู้เชี่ยวชาญ (Human) และระบบอัตโนมัติ (AI) โดยจำแนกตามเกณฑ์ทั้ง 3 ด้านครับ

---

### 1. เกณฑ์ Reference (การอ้างอิงกฎหมาย)

**จุดที่เห็นตรงกัน:**
*   ทั้ง Human และ AI เห็นตรงกันว่า **"มาตราที่อ้างอิงไม่ครบถ้วน"** ตามธงคำตอบ (Expected)
*   ทั้งคู่ระบุว่า Output ขาดมาตราสำคัญเกี่ยวกับการตั้งผู้จัดการมรดก (เช่น ม.1713, 1718, 1719)

**จุดที่ขัดแย้งกัน:**
*   **ประเด็นการจัดรูปแบบ (Formatting):** Human ให้ความสำคัญกับ UX/UI โดยหักคะแนนเรื่องการจัดหน้าจอที่แสดงผลมาตราหลายวรรคติดกัน ทำให้ผู้อ่านสับสน ในขณะที่ AI ไม่ได้ตรวจจับหรือให้ความเห็นเรื่องรูปแบบการนำเสนอ (Format) มุ่งเน้นไปที่เนื้อหาเลขมาตราเพียงอย่างเดียว
*   **การให้คะแนน:** Human ให้คะแนน 0.5 (ให้ค่าความพยายามหรือความถูกต้องบางส่วน) แต่ AI ให้คะแนน 0 (ตัดคะแนนทั้งหมดเพราะขาดมาตราหลักในเฉลย)

**บทวิเคราะห์ความเหมาะสม:**
*   **Human ประเมินได้เหมาะส

## 5. Analyze Score Differences (Overall)

In [10]:
analysis = reviewer.analyze_score_differences(reviews)
print(analysis)

นี่คือรายงานการวิเคราะห์คุณภาพระบบประเมินผลอัตโนมัติ (Auto Evaluation) สำหรับ RAG ทางกฎหมาย โดยเปรียบเทียบกับผู้เชี่ยวชาญ (Human) จากข้อมูล 5 กรณีศึกษาข้างต้น

---

# รายงานวิเคราะห์คุณภาพระบบประเมินผลอัตโนมัติ (AI vs Human)

## 1. อัตราความสอดคล้องรายเกณฑ์ (Consistency Analysis)

จากการวิเคราะห์ข้อมูลทั้ง 5 Entry พบว่าความสอดคล้องของคะแนนระหว่าง AI และ Human มีลักษณะดังนี้:

*   **เกณฑ์ Reference (การอ้างอิง):** **ความสอดคล้องระดับปานกลาง**
    *   คะแนนมักไปในทิศทางเดียวกันในกรณีที่ "มาตราขาดหายไป"
    *   *ความต่าง:* AI แม่นยำเรื่องเนื้อหา (Content) ว่ามาตราใดขาด/เกิน ในขณะที่ Human ให้ความสำคัญกับรูปแบบการจัดวาง (Formatting/UX) มากจนบางครั้งบดบังความถูกต้องของเนื้อหา (เช่น Entry 5 ที่ Human หักคะแนนเพราะการเว้นวรรคผิด)
*   **เกณฑ์ Judgement (การวินิจฉัย):** **ความสอดคล้องระดับต่ำ (Critical Divergence)**
    *   เป็นเกณฑ์ที่มีความขัดแย้งสูงที่สุด AI มักตัดสินแบบ "ขาว/ดำ" (Binary) คือถ้าธงคำตอบผิด = 0 ทันที (Entry 1, 2) ในขณะที่ Human มีการให้คะแนนบางส่วน (Partial Credit)
    *   *จุ

## 6. Suggest Prompt Edits per Scorer

In [11]:
ref_suggestions = reviewer.suggest_prompt_edits(reviews, "reference")
print(ref_suggestions)

จากการวิเคราะห์ข้อมูลเปรียบเทียบระหว่างคะแนนจากระบบ (Auto) และคะแนนจากมนุษย์ (Human) ทั้ง 5 รายการ พบความแตกต่างที่มีนัยสำคัญในเรื่อง "เกณฑ์การให้คะแนน" โดยเฉพาะมิติด้านรูปแบบ (Formatting) และความเคร่งครัดในการอ้างอิงข้อกฎหมายที่ไม่เกี่ยวข้อง นี่คือข้อเสนอแนะในการปรับปรุง Prompt ครับ

---

## 1. สรุปปัญหาที่พบ (Summary of Problems)

1.  **AI มองข้ามเรื่องการจัดรูปแบบ (Formatting Blindness):** ใน Entry 1, 3 และ 5 มนุษย์หักคะแนนหรือให้คะแนนเพียงครึ่งเดียวแม้ว่าเลขมาตราจะถูกต้อง เพราะ "การแสดงผลรวมย่อหน้า/ไม่แยกวรรค" ซึ่ง AI ปัจจุบันไม่ได้ตรวจจับเรื่องนี้เลย (AI ดูแค่เลขมาตราตรงหรือไม่)
2.  **AI ไม่ตรวจสอบความครบถ้วนของเนื้อหาตัวบท (Incomplete Text):** ใน Entry 2 มนุษย์ระบุว่า "ใส่ข้อความมาไม่ครบ" (เช่น ขาดวรรค 2) แต่ AI ให้คะแนนเท่ากันโดยไม่ได้ตัดคะแนนในจุดนี้ชัดเจน
3.  **AI อะลุ่มอล่วยกับมาตราที่ไม่เกี่ยวข้องมากเกินไป (Leninecy on Irrelevant Citations):** ใน Entry 4 AI ให้คะแนน 0.5 เพราะมีมาตราถูกปนอยู่บ้าง แต่มนุษย์ให้ 0 เพราะมีการยกมาตราที่ไม่เกี่ยวข้องเลยมาปน ทำให้คำตอบสับสน (Halluci

In [12]:
judgement_suggestions = reviewer.suggest_prompt_edits(reviews, "judgement")
print(judgement_suggestions)

จากการวิเคราะห์ข้อมูลความแตกต่างระหว่างคะแนนของ AI (Auto Score) และมนุษย์ (Human Score) พบว่ามีช่องว่างสำคัญในด้านการตีความบริบทความเสี่ยงทางธุรกิจ (ธนาคาร), ความลึกของการตรวจสอบตรรกะกฎหมาย (Legal Logic), และประเด็นเรื่องรูปแบบการนำเสนอ (UX/Formatting) ซึ่ง Prompt ปัจจุบันยังไม่ครอบคลุม

นี่คือข้อเสนอแนะในการปรับปรุง Prompt ครับ

---

## 1. สรุปปัญหาที่พบ

1.  **ขาดมิติเรื่องรูปแบบและการนำเสนอ (Formatting & UX):** มนุษย์ให้ความสำคัญกับการจัดย่อหน้าของตัวบทกฎหมาย (การแยกวรรค) มาก (พบใน Entry 1, 3, 5) แต่ Prompt ปัจจุบันไม่ได้สั่งให้ AI ตรวจสอบเรื่องนี้ ทำให้ AI ให้คะแนนเต็มในขณะที่มนุษย์หักคะแนน
2.  **การตรวจสอบตรรกะกฎหมายยังไม่ลึกพอ (Superficial Logic Check):** ใน Entry 4 AI ให้คะแนนเต็มเพราะมีการอ้างมาตราถูกต้องและระบุผลลัพธ์ (โมฆียะ) ถูกต้องตามคีย์เวิร์ด แต่พลาดประเด็นสำคัญคือ "การปรับบทกฎหมายกับข้อเท็จจริง" (Application of Law) โดยเฉพาะเรื่องข้อยกเว้น ทำให้ผลสรุปผิดเพี้ยนไปจากความจริง
3.  **ความเข้มงวดเรื่องผลลัพธ์ vs ความเสี่ยงทางธุรกิจ:** ใน Entry 1 AI ตัดสินว่าผิดทันทีเพราะผลลัพธ

In [13]:
suggestion_suggestions = reviewer.suggest_prompt_edits(reviews, "suggestion")
print(suggestion_suggestions)

จากการวิเคราะห์ Prompt ปัจจุบันเทียบกับข้อมูลความขัดแย้ง (Discrepancy) ระหว่าง AI และมนุษย์ พบประเด็นสำคัญที่ทำให้คะแนนไม่ตรงกัน และขอเสนอแนวทางปรับปรุงดังนี้ครับ

---

## 1. สรุปปัญหาที่พบ

1.  **AI ตัดสินด้วย "ทฤษฎีกฎหมาย" แต่มนุษย์ตัดสินด้วย "นโยบายธนาคาร/ความเสี่ยง":**
    *   ใน Entry 1 AI ให้คะแนนเต็มเพราะคำตอบถูกต้องตามประมวลกฎหมายแพ่ง (ป.พ.พ.) แต่ Human ให้ 0 เพราะในทางปฏิบัติธนาคารต้องการ "ปิดความเสี่ยง" จึงต้องให้ตั้งผู้จัดการมรดกเท่านั้น AI ไม่เข้าใจบริบทว่า "ระเบียบปฏิบัติ (Policy)" สำคัญกว่า "สิทธิตามกฎหมายทั่วไป" ในบริบทนี้
2.  **AI พยายาม "เถียง" เฉลย (EXPECTED):**
    *   ใน Entry 1 AI ระบุว่า OUTPUT ดีกว่า EXPECTED เพราะอ้างกฎหมายใหม่กว่า แต่ระบบ Scorer ที่ดีต้องยึด EXPECTED เป็น "ความถูกต้องสัมบูรณ์" (Ground Truth) ในบริบทของการตรวจข้อสอบหรือตรวจสอบความถูกต้องตามระเบียบองค์กร ไม่ใช่เวทีถกเถียงทางวิชาการ
3.  **ขาดมิติเรื่อง "ความสามารถในการนำไปใช้ปฏิบัติจริง" (Actionability):**
    *   ใน Entry 3 Human ต้องการคำแนะนำที่เจ้าหน้าที่หน้างานนำไปบอกลูกค้าต่อได้เลย (เช่น แนะ

## 7. Full Pipeline (Alternative)
Run everything in one call using `run_full_review`.

In [ ]:
# report = reviewer.run_full_review(data)
#
# print("=== Overall Score Analysis ===")
# print(report.overall_score_analysis)
#
# for scorer_name, suggestion in report.prompt_suggestions.items():
#     print(f"\n=== Prompt Suggestions: {scorer_name} ===")
#     print(suggestion)